In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
#read the csv from the training path and print first five rows
data_train = pd.read_csv("/content/drive/MyDrive/1:1_Krish_Sathish/Data/processed_data_crossvalidation/train.csv")
print(data_train.head)

<bound method NDFrame.head of          Unnamed: 0  flow_duration  Header_Length  Protocol Type  Duration  \
0                 0       0.845634       2.472001      -1.040490 -0.438100   
1                 1      -0.155719       0.552384      -0.603546 -0.438100   
2                 2      -0.200352      -0.567503       1.075569 -0.385350   
3                 3       1.382824      -0.486753      -1.199379 -0.515430   
4                 4      -0.158543       1.653662      -0.603546 -0.438100   
...             ...            ...            ...            ...       ...   
2985850     2985850      -0.093464      -0.561845      -0.915649  1.072595   
2985851     2985851      -0.197028      -0.404155      -0.609788 -0.385350   
2985852     2985852      -0.192612      -0.560556      -1.227752 -0.438100   
2985853     2985853      -0.200528      -0.567505       0.968886 -0.462956   
2985854     2985854      -0.200528      -0.567506       1.098834 -0.438100   

             Rate     Srate  fin_

In [3]:
LABEL_COLUMN = "label"

In [4]:
# labels are there as string otherwise words
# need to convert the labels into numbers
print(data_train[LABEL_COLUMN].value_counts())
LABELS = list(data_train[LABEL_COLUMN].unique())
# sort the labels
LABELS.sort()
print(LABELS)

label
BenignTraffic         878556
Mirai-greeth_flood    793493
Mirai-udpplain        712461
Mirai-greip_flood     601345
Name: count, dtype: int64
['BenignTraffic', 'Mirai-greeth_flood', 'Mirai-greip_flood', 'Mirai-udpplain']


In [5]:
# convert into numbers
data_train[LABEL_COLUMN] = pd.factorize(data_train[LABEL_COLUMN], sort = True)[0]
data_train[LABEL_COLUMN].value_counts()

,count
label,
0,878556
1,793493
3,712461
2,601345


In [6]:
# get all features without the labels
# all the rows
# all the columns without the last column
X = data_train.drop(LABEL_COLUMN, axis = 1).values
# all rows
# only the lastb column, which is the label
Y = data_train[LABEL_COLUMN].values

In [7]:
import threading, time

def _keep_alive():
    while True:
        time.sleep(60)
        try:
            from google.colab import output
            output.eval_js("0")
        except: pass

threading.Thread(target=_keep_alive, daemon=True).start()
print("Keep-alive started")

Keep-alive started


In [8]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score
import numpy as np

In [9]:
def cross_validation(model, data = (X, Y), splits = 3):
    kf = KFold(n_splits=splits, shuffle=True, random_state=42)
    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []
    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]
        # Fit the defined model
        model.fit(X_train, y_train)
        # Make predictions on the test data
        y_pred = model.predict(X_valid)
        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_pred, y_valid))
        precision.append(precision_score(y_pred, y_valid, average = 'macro'))
        recall.append(recall_score(y_pred, y_valid, average = 'macro'))
    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)
    print("Mean Accuracy: {}".format(accuracy_set.mean()))
    print("Mean Precision: {}".format(precision_set.mean()))
    print("Mean Recall: {}".format(recall_set.mean()))



In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
knn = KNeighborsClassifier(n_neighbors=1)
print("\nNeighbors: 1")
cross_validation(knn)


Neighbors: 1


In [ ]:
#MLP
from sklearn.neural_network import MLPClassifier
lr = [0.000001,0.00001,0.0001,0.001,0.005,0.01, 0.05]
ep = [10,20,30,40,50]
for l in lr:
  for epoch in ep:
    print("\n Learning Rate:", l," epoch:", epoch)

    mlp = MLPClassifier(learning_rate_init=l, max_iter=epoch)
    cross_validation(mlp)



 Learning Rate: 0.001  epoch: 10


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(


Mean Accuracy: 0.6534670973640716
Mean Precision: 0.6031020978226632
Mean Recall: 0.6693252985953592

 Learning Rate: 0.005  epoch: 10


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/lo

Mean Accuracy: 0.29424938585430305
Mean Precision: 0.2500103439041052
Mean Recall: 0.4624496598676477

 Learning Rate: 0.01  epoch: 10


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters
max_depth = [6,7]
for depth in max_depth:
  print("\n trees:",80," depth:", depth)
  rf = RandomForestClassifier(n_estimators=80, max_depth= depth, n_jobs = -1)
  cross_validation(rf)


 trees: 80  depth: 6
Mean Accuracy: 0.994700010549742
Mean Precision: 0.9943166559461835
Mean Recall: 0.9951352846623255

 trees: 80  depth: 7
Mean Accuracy: 0.9967004425867967
Mean Precision: 0.9964063732673155
Mean Recall: 0.9971119023724487


In [ ]:
from lightgbm import LGBMClassifier
depths = [1,2,3,4,5,6,7,8,9,10,11,12]
n_estimators_list = [10,20,30,40,50,60,70,80,90,100]
for depth in depths:
    for trees in n_estimators_list:
            print("\n" + "-"*50)
            print(f"Running LGBM with max_depth={depth}, n_estimators={trees}")
            print("-"*50)
            lgbm = LGBMClassifier(max_depth=depth,n_estimators=trees,learning_rate=0.001,verbosity=-1)
            cross_validation(lgbm)

In [ ]:
from xgboost import XGBClassifier
depths = [1,2,3,4,5,6,7,8,9,10,11,12]
n_estimators_list = [10,20,30,40,50,60,70,80,90,100]
for depth in depths:
    for trees in n_estimators_list:
            print("\n" + "-"*50)
            print(f"Running XGBoost with max_depth={depth}, n_estimators={trees}")
            print("-"*50)
            xgb = XGBClassifier(max_depth=depth,n_estimators=trees,learning_rate=0.001)
            cross_validation(xgb)